In [3]:
import numpy as np


class KNN:

    def __init__(self, k=None, k_selection="sqrt", max_k=25, folds=5):

        self.k = k
        self.k_selection = k_selection
        self.max_k = max_k
        self.folds = folds

    def fit(self, X, y):

        self.X_train = np.array(X)
        self.y_train = np.array(y)

        if self.k_selection == "sqrt":

            self.k = int(np.sqrt(len(self.X_train)))

            if self.k % 2 == 0:
                self.k += 1

            print(f"Using sqrt(N) heuristic")
            print(f"Selected K = {self.k}")

        elif self.k_selection == "cv":

            self.k = self._cross_validate()

            print(f"Best K from Cross Validation = {self.k}")

        elif self.k_selection == "manual":

            if self.k is None:
                raise ValueError("Provide k when using manual mode")

            print(f"Using Manual K = {self.k}")

    def euclidean_distance(self, x1, x2):

        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict_single(self, x):

        distances = []

        for x_train in self.X_train:

            distance = self.euclidean_distance(x, x_train)

            distances.append(distance)

        distances = np.array(distances)

        k_indices = np.argsort(distances)[:self.k]

        k_labels = self.y_train[k_indices]

        unique_labels, counts = np.unique(
            k_labels,
            return_counts=True
        )

        return unique_labels[np.argmax(counts)]

    def predict(self, X):

        predictions = []

        for x in X:

            predictions.append(
                self.predict_single(x)
            )

        return np.array(predictions)

    def accuracy(self, y_true, y_pred):

        return np.mean(y_true == y_pred)

    def _cross_validate(self):

        X = self.X_train.copy()
        y = self.y_train.copy()

        n_samples = len(X)

        indices = np.arange(n_samples)

        np.random.shuffle(indices)

        X = X[indices]
        y = y[indices]

        fold_size = n_samples // self.folds

        best_k = 1
        best_accuracy = 0

        print("\nCross Validation Results")

        for k in range(1, self.max_k + 1, 2):

            fold_accuracies = []

            for fold in range(self.folds):

                start = fold * fold_size

                if fold == self.folds - 1:
                    end = n_samples
                else:
                    end = start + fold_size

                X_val = X[start:end]
                y_val = y[start:end]

                X_train = np.concatenate(
                    (X[:start], X[end:]),
                    axis=0
                )

                y_train = np.concatenate(
                    (y[:start], y[end:]),
                    axis=0
                )

                correct = 0

                for i in range(len(X_val)):

                    distances = []

                    for train_sample in X_train:

                        dist = np.sqrt(
                            np.sum(
                                (X_val[i] - train_sample) ** 2
                            )
                        )

                        distances.append(dist)

                    distances = np.array(distances)

                    k_indices = np.argsort(distances)[:k]

                    k_labels = y_train[k_indices]

                    unique_labels, counts = np.unique(
                        k_labels,
                        return_counts=True
                    )

                    prediction = unique_labels[
                        np.argmax(counts)
                    ]

                    if prediction == y_val[i]:
                        correct += 1

                fold_accuracy = correct / len(X_val)

                fold_accuracies.append(
                    fold_accuracy
                )

            avg_accuracy = np.mean(
                fold_accuracies
            )

            print(
                f"K = {k} | CV Accuracy = {avg_accuracy:.4f}"
            )

            if avg_accuracy > best_accuracy:

                best_accuracy = avg_accuracy
                best_k = k

        return best_k

In [4]:
import pandas as pd
import numpy as np

# Toy Dataset

df = pd.DataFrame({
    "StudyHours": [1,2,3,4,5,6,7,8,9,10,2,3,5,7,8],
    "Attendance": [50,55,60,65,70,75,80,85,90,95,58,62,72,82,88],
    "Pass": [0,0,0,0,1,1,1,1,1,1,0,0,1,1,1]
})

print("Dataset:\n")
print(df)

# Features and Target

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Feature Scaling (recommended for KNN)

X = (X - X.mean(axis=0)) / X.std(axis=0)

# Shuffle

np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

# Train-Test Split

split = int(0.8 * len(X))

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

print("\nTraining Samples:", len(X_train))
print("Testing Samples :", len(X_test))

# Create Model

model = KNN(
    k_selection="cv",
    max_k=7,
    folds=3
)

# Train

model.fit(X_train, y_train)

# Predict

y_pred = model.predict(X_test)

# Accuracy

accuracy = model.accuracy(
    y_test,
    y_pred
)

print("\nSelected K =", model.k)
print("Test Accuracy =", accuracy)

# Actual vs Predicted

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

print("\nPredictions:")
print(results)

Dataset:

    StudyHours  Attendance  Pass
0            1          50     0
1            2          55     0
2            3          60     0
3            4          65     0
4            5          70     1
5            6          75     1
6            7          80     1
7            8          85     1
8            9          90     1
9           10          95     1
10           2          58     0
11           3          62     0
12           5          72     1
13           7          82     1
14           8          88     1

Training Samples: 12
Testing Samples : 3

Cross Validation Results
K = 1 | CV Accuracy = 1.0000
K = 3 | CV Accuracy = 0.9167
K = 5 | CV Accuracy = 0.6667
K = 7 | CV Accuracy = 0.2500
Best K from Cross Validation = 1

Selected K = 1
Test Accuracy = 1.0

Predictions:
   Actual  Predicted
0       1          1
1       0          0
2       1          1
